# 🧹 NSEI Stock Data — Uncleaning & Cleaning Pipeline

This notebook demonstrates a **complete data quality workflow** on the NSEI (NSE India) featured dataset.

**Structure:**
1. Load original data & baseline inspection
2. **Introduce artificial dirtiness** (missing values, duplicates, outliers, type errors, inconsistencies)
3. **Clean step-by-step** with documentation at every stage
4. Final validation & summary report

---

## 📦 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('✅ All libraries imported successfully')

## 📂 1. Load Original Data & Baseline Inspection

In [ ]:
df_original = pd.read_csv('NSEI_Featured.csv')

print(f'Shape: {df_original.shape}')
print(f'Rows: {df_original.shape[0]:,}  |  Columns: {df_original.shape[1]}')
df_original.head(3)

In [ ]:
print('=== COLUMN DTYPES ===')
print(df_original.dtypes)
print('\n=== MISSING VALUES (original) ===')
print(df_original.isnull().sum().sum(), 'total nulls')
print('\n=== DUPLICATES (original) ===')
print(df_original.duplicated().sum(), 'duplicate rows')

In [ ]:
print('=== DESCRIPTIVE STATISTICS (Original) ===')
df_original.describe().T[['min', 'max', 'mean', 'std', '25%', '50%', '75%']].head(20)

In [ ]:
# Visualise original Close price
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df_original['Close'].plot(ax=axes[0], color='steelblue', linewidth=0.8)
axes[0].set_title('NSEI Close Price (Original)', fontsize=13)
axes[0].set_xlabel('Index'); axes[0].set_ylabel('Price')

df_original['Volume'].plot(ax=axes[1], color='coral', linewidth=0.8)
axes[1].set_title('NSEI Volume (Original)', fontsize=13)
axes[1].set_xlabel('Index'); axes[1].set_ylabel('Volume')
plt.tight_layout()
plt.show()

---

## 🦠 2. Introduce Artificial Dirtiness

We deliberately corrupt the data to simulate real-world data quality problems:

| # | Problem Type | Description |
|---|---|---|
| 1 | **Missing Values (MCAR)** | Random NaNs in numeric columns |
| 2 | **Missing Values (MAR)** | NaNs in RSI when Volume is low |
| 3 | **Duplicate Rows** | Randomly repeat some rows |
| 4 | **Wrong Dtypes** | Date stored as generic string, Volume as float with `.0` |
| 5 | **Outliers / Extreme Values** | Inject unrealistic price and RSI spikes |
| 6 | **Inconsistent Categories** | Mixed-case / whitespace in string-like columns |
| 7 | **Negative Prices** | Corrupt a few Close values to negative |
| 8 | **OHLC Violations** | High < Low for some rows |
| 9 | **Whitespace in Column Names** | Add trailing spaces to some headers |
| 10 | **Out-of-Range Values** | RSI values > 100 or < 0 |

In [ ]:
np.random.seed(42)
df_dirty = df_original.copy()

n = len(df_dirty)

# ── 1. MCAR: randomly null ~5% of numeric columns ──────────────────────────
mcar_cols = ['Close', 'Open', 'High', 'Low', 'MA_5', 'MA_20', 'Volatility_10',
             'Volume_Spike', 'BB_Position', 'ROC_5', 'ROC_20']
for col in mcar_cols:
    null_idx = np.random.choice(n, size=int(n * 0.05), replace=False)
    df_dirty.loc[null_idx, col] = np.nan

print(f'1️⃣  MCAR nulls injected: {df_dirty[mcar_cols].isnull().sum().sum()}')

In [ ]:
# ── 2. MAR: null RSI_14 when Volume is in bottom 10th percentile ────────────
low_vol_mask = df_dirty['Volume'] < df_dirty['Volume'].quantile(0.10)
df_dirty.loc[low_vol_mask, 'RSI_14'] = np.nan
print(f'2️⃣  MAR nulls in RSI_14 (low-volume rows): {df_dirty["RSI_14"].isnull().sum()}')

In [ ]:
# ── 3. Duplicate rows ────────────────────────────────────────────────────────
dup_idx = np.random.choice(n, size=80, replace=False)
df_dirty = pd.concat([df_dirty, df_dirty.iloc[dup_idx]], ignore_index=True)
print(f'3️⃣  Duplicates injected → new shape: {df_dirty.shape}, dupes: {df_dirty.duplicated().sum()}')

In [ ]:
# ── 4. Wrong dtypes: Volume as float string, Date as raw string ──────────────
df_dirty['Volume'] = df_dirty['Volume'].astype(float)   # float instead of int
# Date is already a string from CSV; we add extra whitespace to simulate messiness
df_dirty['Date'] = df_dirty['Date'].astype(str).str.strip() + '  '  # trailing spaces
print(f'4️⃣  Dtypes corrupted — Volume: {df_dirty["Volume"].dtype}, Date sample: "{df_dirty["Date"].iloc[0]}"')

In [ ]:
# ── 5. Outliers / extreme values ─────────────────────────────────────────────
out_idx = np.random.choice(len(df_dirty), size=30, replace=False)
df_dirty.loc[out_idx[:15], 'Close'] = df_dirty['Close'].mean() * np.random.uniform(5, 10, 15)
df_dirty.loc[out_idx[15:], 'Volume'] = df_dirty['Volume'].mean() * np.random.uniform(20, 50, 15)
print(f'5️⃣  Outliers injected — max Close: {df_dirty["Close"].max():,.1f}, max Volume: {df_dirty["Volume"].max():,.0f}')

In [ ]:
# ── 6. Inconsistent Target_Strong: mix -1, 0, 1 with strings 'yes'/'no' ──────
str_idx = np.random.choice(len(df_dirty), size=50, replace=False)
df_dirty['Target_Strong'] = df_dirty['Target_Strong'].astype(object)
df_dirty.loc[str_idx[:25], 'Target_Strong'] = '  yes '
df_dirty.loc[str_idx[25:], 'Target_Strong'] = 'NO'
print(f'6️⃣  Inconsistent categories in Target_Strong: {df_dirty["Target_Strong"].unique()[:8]}')

In [ ]:
# ── 7. Negative prices ───────────────────────────────────────────────────────
neg_idx = np.random.choice(len(df_dirty), size=20, replace=False)
df_dirty.loc[neg_idx, 'Close'] = -abs(df_dirty.loc[neg_idx, 'Close'])
print(f'7️⃣  Negative Close values: {(df_dirty["Close"] < 0).sum()}')

In [ ]:
# ── 8. OHLC violations: High < Low ───────────────────────────────────────────
ohlc_idx = np.random.choice(len(df_dirty), size=25, replace=False)
df_dirty.loc[ohlc_idx, ['High', 'Low']] = df_dirty.loc[ohlc_idx, ['Low', 'High']].values
violations = (df_dirty['High'] < df_dirty['Low']).sum()
print(f'8️⃣  OHLC violations (High < Low): {violations}')

In [ ]:
# ── 9. Whitespace in column names ────────────────────────────────────────────
rename_map = {c: c + '  ' for c in ['RSI_14', 'MACD', 'BB_Width', 'Range']}
df_dirty.rename(columns=rename_map, inplace=True)
print(f'9️⃣  Dirty column names (sample): {[c for c in df_dirty.columns if c.endswith("  ")]}')

In [ ]:
# ── 10. Out-of-range RSI (RSI must be 0-100) ──────────────────────────────────
rsi_col = 'RSI_14  '
oor_idx = np.random.choice(len(df_dirty), size=40, replace=False)
df_dirty.loc[oor_idx[:20], rsi_col] = np.random.uniform(105, 150, 20)
df_dirty.loc[oor_idx[20:], rsi_col] = np.random.uniform(-50, -1, 20)
oor_count = ((df_dirty[rsi_col] > 100) | (df_dirty[rsi_col] < 0)).sum()
print(f'🔟  Out-of-range RSI values: {oor_count}')

print(f'\n📊 DIRTY DATASET SUMMARY')
print(f'   Shape         : {df_dirty.shape}')
print(f'   Total Nulls   : {df_dirty.isnull().sum().sum()}')
print(f'   Duplicates    : {df_dirty.duplicated().sum()}')
print(f'   Negative Close: {(df_dirty["Close"] < 0).sum()}')

---

## 🧹 3. Data Cleaning Pipeline

We now apply a systematic, documented cleaning process on `df_dirty`.

> **Convention:** we track a `df_clean` and print before/after counts at each step.

In [ ]:
df_clean = df_dirty.copy()

def snapshot(label):
    print(f'[{label}]  shape={df_clean.shape}  nulls={df_clean.isnull().sum().sum()}  '
          f'dupes={df_clean.duplicated().sum()}')

snapshot('START')

### 🔧 Step 1 — Fix Column Names (strip whitespace)

In [ ]:
before = list(df_clean.columns)
df_clean.columns = df_clean.columns.str.strip()
after = list(df_clean.columns)
changed = [f'"{b}" → "{a}"' for b, a in zip(before, after) if b != a]
print(f'Column names fixed: {len(changed)}')
for c in changed:
    print(' ', c)
snapshot('After Step 1')

### 🔧 Step 2 — Remove Duplicate Rows

In [ ]:
before_n = len(df_clean)
df_clean.drop_duplicates(inplace=True)
df_clean.reset_index(drop=True, inplace=True)
removed = before_n - len(df_clean)
print(f'Duplicate rows removed: {removed}')
snapshot('After Step 2')

### 🔧 Step 3 — Fix Data Types

In [ ]:
# 3a. Date → datetime (strip whitespace first)
df_clean['Date'] = df_clean['Date'].astype(str).str.strip()
df_clean['Date'] = pd.to_datetime(df_clean['Date'], utc=True, errors='coerce')
print(f'Date dtype: {df_clean["Date"].dtype}  |  NaT count: {df_clean["Date"].isnull().sum()}')

# 3b. Volume → int (after we handle nulls later; keep as float for now but note intent)
#     Will cast to Int64 (nullable int) after null imputation
df_clean['Volume'] = pd.to_numeric(df_clean['Volume'], errors='coerce')
print(f'Volume dtype: {df_clean["Volume"].dtype}')

# 3c. Target_Strong: standardise to -1 / 0 / 1
ts_map = {'  yes ': 1, 'YES': 1, 'yes': 1, 'NO': -1, 'no': -1, '  no ': -1}
df_clean['Target_Strong'] = df_clean['Target_Strong'].apply(
    lambda x: ts_map.get(str(x).strip().lower(), x) if isinstance(x, str) else x
)
df_clean['Target_Strong'] = pd.to_numeric(df_clean['Target_Strong'], errors='coerce')
print(f'Target_Strong unique values: {sorted(df_clean["Target_Strong"].dropna().unique())}')

snapshot('After Step 3')

### 🔧 Step 4 — Handle OHLC Violations (High < Low)

In [ ]:
# Strategy: swap High and Low where High < Low
violation_mask = df_clean['High'] < df_clean['Low']
n_violations = violation_mask.sum()
print(f'OHLC violations before fix: {n_violations}')

df_clean.loc[violation_mask, ['High', 'Low']] = (
    df_clean.loc[violation_mask, ['Low', 'High']].values
)

remaining = (df_clean['High'] < df_clean['Low']).sum()
print(f'OHLC violations after fix : {remaining}')
snapshot('After Step 4')

### 🔧 Step 5 — Handle Negative Prices

In [ ]:
price_cols = ['Open', 'High', 'Low', 'Close']

for col in price_cols:
    neg_count = (df_clean[col] < 0).sum()
    if neg_count > 0:
        # Convert negative prices to NaN — they are invalid; we'll impute later
        df_clean.loc[df_clean[col] < 0, col] = np.nan
        print(f'  {col}: {neg_count} negative values → set to NaN')

snapshot('After Step 5')

### 🔧 Step 6 — Detect & Cap Outliers

We use the **IQR method** (1.5 × IQR fence) for `Close` and `Volume`.

In [ ]:
def iqr_bounds(series, factor=3.0):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - factor * iqr, q3 + factor * iqr

outlier_cols = ['Close', 'Open', 'High', 'Low', 'Volume']

fig, axes = plt.subplots(1, len(outlier_cols), figsize=(18, 4))
for ax, col in zip(axes, outlier_cols):
    df_clean[col].dropna().plot(kind='box', ax=ax, color='steelblue')
    ax.set_title(col, fontsize=10)
plt.suptitle('Boxplots BEFORE Outlier Capping', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
for col in outlier_cols:
    lo, hi = iqr_bounds(df_clean[col].dropna())
    n_low  = (df_clean[col] < lo).sum()
    n_high = (df_clean[col] > hi).sum()
    # Cap (Winsorize) outliers
    df_clean[col] = df_clean[col].clip(lower=lo, upper=hi)
    print(f'  {col:<15}  lower_fence={lo:>10.1f}  upper_fence={hi:>12.1f}  '
          f'capped_low={n_low}  capped_high={n_high}')

snapshot('After Step 6')

In [ ]:
fig, axes = plt.subplots(1, len(outlier_cols), figsize=(18, 4))
for ax, col in zip(axes, outlier_cols):
    df_clean[col].dropna().plot(kind='box', ax=ax, color='seagreen')
    ax.set_title(col, fontsize=10)
plt.suptitle('Boxplots AFTER Outlier Capping', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 🔧 Step 7 — Fix Out-of-Range RSI Values

In [ ]:
oor_before = ((df_clean['RSI_14'] > 100) | (df_clean['RSI_14'] < 0)).sum()
print(f'Out-of-range RSI before fix: {oor_before}')

# Set impossible RSI values to NaN — they will be imputed in Step 8
df_clean.loc[(df_clean['RSI_14'] > 100) | (df_clean['RSI_14'] < 0), 'RSI_14'] = np.nan

oor_after = ((df_clean['RSI_14'] > 100) | (df_clean['RSI_14'] < 0)).sum()
print(f'Out-of-range RSI after fix : {oor_after}')
snapshot('After Step 7')

### 🔧 Step 8 — Missing Value Imputation

We use **different strategies** depending on the nature of each column:

| Column Group | Strategy | Reason |
|---|---|---|
| Price cols (OHLC) | Forward-fill then backward-fill | Time series — adjacent values are best proxies |
| Volume | Median imputation | Right-skewed, median more robust |
| RSI_14 | Forward-fill | MAR: low volume → use last known RSI |
| Technical indicators (MA, BB, etc.) | Forward-fill | Calculated from prices, carry-forward is appropriate |
| Return cols | Zero imputation | Missing return = no change assumption |
| Target_Strong | Mode imputation | Categorical — most common class |

In [ ]:
print('=== MISSING VALUES BEFORE IMPUTATION ===')
null_summary = df_clean.isnull().sum()
null_summary = null_summary[null_summary > 0].sort_values(ascending=False)
print(null_summary.to_string())
print(f'\nTotal missing cells: {null_summary.sum()}')

In [ ]:
# ── Heatmap of missing values ────────────────────────────────────────────────
missing_pct = (df_clean.isnull().sum() / len(df_clean) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

plt.figure(figsize=(12, 4))
missing_pct.plot(kind='bar', color='tomato', edgecolor='white')
plt.axhline(5, color='navy', linestyle='--', linewidth=1, label='5% threshold')
plt.title('% Missing Values per Column (Before Imputation)', fontsize=13)
plt.ylabel('% Missing')
plt.xlabel('')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Sort by Date before imputation (time-series logic)
df_clean.sort_values('Date', inplace=True)
df_clean.reset_index(drop=True, inplace=True)

# ── 8a. OHLC: forward-fill then backward-fill ────────────────────────────────
price_cols = ['Open', 'High', 'Low', 'Close']
df_clean[price_cols] = df_clean[price_cols].ffill().bfill()
print(f'OHLC nulls after ffill/bfill: {df_clean[price_cols].isnull().sum().sum()}')

In [ ]:
# ── 8b. Volume: median imputation ────────────────────────────────────────────
vol_median = df_clean['Volume'].median()
n_vol_null = df_clean['Volume'].isnull().sum()
df_clean['Volume'].fillna(vol_median, inplace=True)
print(f'Volume: {n_vol_null} nulls filled with median = {vol_median:,.0f}')

# Now safely cast Volume to integer
df_clean['Volume'] = df_clean['Volume'].round().astype('Int64')
print(f'Volume dtype after cast: {df_clean["Volume"].dtype}')

In [ ]:
# ── 8c. RSI_14: forward-fill (MAR — related to Volume) ───────────────────────
n_rsi_null = df_clean['RSI_14'].isnull().sum()
df_clean['RSI_14'] = df_clean['RSI_14'].ffill().bfill()
print(f'RSI_14: {n_rsi_null} nulls filled with forward-fill')

In [ ]:
# ── 8d. Technical indicators (MA, BB, Volatility, etc.): forward-fill ─────────
tech_cols = ['MA_5', 'MA_20', 'MA_50', 'MA_200', 'Volatility_10', 'Volatility_30',
             'BB_Upper', 'BB_Lower', 'BB_Width', 'BB_Position',
             'Volume_MA_20', 'Volume_Spike', 'Volume_Trend',
             'MACD', 'MACD_Signal', 'MACD_Hist']
tech_cols = [c for c in tech_cols if c in df_clean.columns]

before = df_clean[tech_cols].isnull().sum().sum()
df_clean[tech_cols] = df_clean[tech_cols].ffill().bfill()
after = df_clean[tech_cols].isnull().sum().sum()
print(f'Technical indicators: {before} nulls → {after} after ffill/bfill')

In [ ]:
# ── 8e. Return columns: fill with 0 (no movement assumed) ────────────────────
return_cols = [c for c in df_clean.columns if 'Return' in c or 'ROC' in c]
before = df_clean[return_cols].isnull().sum().sum()
df_clean[return_cols] = df_clean[return_cols].fillna(0)
print(f'Return/ROC columns: {before} nulls filled with 0')

In [ ]:
# ── 8f. Target_Strong: mode imputation ───────────────────────────────────────
mode_val = df_clean['Target_Strong'].mode()[0]
n_ts_null = df_clean['Target_Strong'].isnull().sum()
df_clean['Target_Strong'].fillna(mode_val, inplace=True)
df_clean['Target_Strong'] = df_clean['Target_Strong'].astype(int)
print(f'Target_Strong: {n_ts_null} nulls filled with mode = {mode_val}')

In [ ]:
# ── Any remaining nulls → median for numeric, ffill for others ───────────────
remaining_nulls = df_clean.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
if len(remaining_nulls) > 0:
    print('Remaining nulls — applying fallback median/ffill:')
    for col in remaining_nulls.index:
        if pd.api.types.is_numeric_dtype(df_clean[col]):
            df_clean[col].fillna(df_clean[col].median(), inplace=True)
        else:
            df_clean[col].ffill(inplace=True)
    print(f'  Resolved {remaining_nulls.sum()} additional nulls')
else:
    print('✅ No remaining nulls after all imputation steps')

snapshot('After Step 8')

### 🔧 Step 9 — Sort & Reset Index

In [ ]:
df_clean.sort_values('Date', inplace=True)
df_clean.reset_index(drop=True, inplace=True)
print(f'Sorted by Date. Index range: 0 – {len(df_clean)-1}')
snapshot('After Step 9')

### 🔧 Step 10 — Final Data Type Enforcement

In [ ]:
# Integer binary columns
int_cols = ['Target_Direction', 'Target_Strong', 'High_Vol_Regime', 'RSI_Extreme', 'MACD_Positive']
int_cols = [c for c in int_cols if c in df_clean.columns]
for col in int_cols:
    df_clean[col] = df_clean[col].astype(int)

# Float columns (all remaining numeric except Volume and int_cols)
float_cols = df_clean.select_dtypes(include='number').columns.difference(int_cols + ['Volume'])
df_clean[float_cols] = df_clean[float_cols].astype(float)

print('Final dtypes:')
print(df_clean.dtypes.value_counts())
snapshot('After Step 10')

---

## ✅ 4. Validation & Comparison Report

In [ ]:
print('═' * 60)
print(f'  {"METRIC":<30} {"DIRTY":>12} {"CLEAN":>12}')
print('═' * 60)
print(f'  {"Rows":<30} {len(df_dirty):>12,} {len(df_clean):>12,}')
print(f'  {"Total Null Cells":<30} {df_dirty.isnull().sum().sum():>12,} {df_clean.isnull().sum().sum():>12,}')
print(f'  {"Duplicate Rows":<30} {df_dirty.duplicated().sum():>12,} {df_clean.duplicated().sum():>12,}')
print(f'  {"Negative Close":<30} {(df_dirty["Close"] < 0).sum():>12,} {(df_clean["Close"] < 0).sum():>12,}')
vio_dirty = (df_dirty['High'] < df_dirty['Low']).sum()
vio_clean = (df_clean['High'] < df_clean['Low']).sum()
print(f'  {"OHLC Violations":<30} {vio_dirty:>12,} {vio_clean:>12,}')
rsi_oor_dirty = ((df_dirty['RSI_14  '] > 100) | (df_dirty['RSI_14  '] < 0)).sum()
rsi_oor_clean = ((df_clean['RSI_14'] > 100) | (df_clean['RSI_14'] < 0)).sum()
print(f'  {"RSI Out-of-Range":<30} {rsi_oor_dirty:>12,} {rsi_oor_clean:>12,}')
print(f'  {"Columns w/ Whitespace Names":<30} {sum(c != c.strip() for c in df_dirty.columns):>12} {sum(c != c.strip() for c in df_clean.columns):>12}')
print('═' * 60)

In [ ]:
# Missing value heatmap: after cleaning
missing_after = df_clean.isnull().sum()
print(f'Missing values after cleaning: {missing_after.sum()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
missing_before_plot = (df_dirty.isnull().sum() / len(df_dirty) * 100).sort_values(ascending=False).head(15)
missing_before_plot.plot(kind='bar', ax=axes[0], color='tomato', edgecolor='white')
axes[0].set_title('% Missing (Dirty) — Top 15 Columns', fontsize=11)
axes[0].set_ylabel('% Missing')
axes[0].tick_params(axis='x', rotation=45)

missing_after_plot = (df_clean.isnull().sum() / len(df_clean) * 100).sort_values(ascending=False).head(15)
if missing_after_plot.sum() == 0:
    axes[1].bar(['All columns'], [0], color='seagreen')
    axes[1].set_ylim(0, 1)
    axes[1].text(0, 0.5, '✅ 0% missing', ha='center', va='center', fontsize=14, color='seagreen')
else:
    missing_after_plot.plot(kind='bar', ax=axes[1], color='seagreen', edgecolor='white')
axes[1].set_title('% Missing (Clean) — Top 15 Columns', fontsize=11)
axes[1].set_ylabel('% Missing')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution comparison: Close price
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_dirty['Close'].dropna().plot(kind='hist', bins=60, ax=axes[0], color='tomato',
                                 edgecolor='white', alpha=0.85)
axes[0].set_title('Close Price Distribution — DIRTY', fontsize=12)
axes[0].set_xlabel('Close')

df_clean['Close'].plot(kind='hist', bins=60, ax=axes[1], color='steelblue',
                       edgecolor='white', alpha=0.85)
axes[1].set_title('Close Price Distribution — CLEAN', fontsize=12)
axes[1].set_xlabel('Close')
plt.tight_layout()
plt.show()

In [ ]:
# RSI distribution before/after
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df_dirty['RSI_14  '].dropna().plot(kind='hist', bins=50, ax=axes[0],
                                    color='tomato', edgecolor='white', alpha=0.85)
axes[0].axvline(0,   color='black', linewidth=1, linestyle='--', label='0')
axes[0].axvline(100, color='black', linewidth=1, linestyle='--', label='100')
axes[0].set_title('RSI_14 Distribution — DIRTY', fontsize=12)
axes[0].legend()

df_clean['RSI_14'].plot(kind='hist', bins=50, ax=axes[1],
                        color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(0,   color='black', linewidth=1, linestyle='--')
axes[1].axvline(100, color='black', linewidth=1, linestyle='--')
axes[1].set_title('RSI_14 Distribution — CLEAN', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print('=== CLEAN DATASET — FINAL DESCRIPTIVE STATISTICS ===')
df_clean.describe().T[['min', 'max', 'mean', 'std']].head(20)

In [ ]:
print('=== CLEAN DATASET — SAMPLE ROWS ===')
df_clean.head(5)

---

## 💾 5. Save Clean Dataset

In [ ]:
df_clean.to_csv('NSEI_Featured_Clean.csv', index=False)
print('✅ Clean dataset saved to: NSEI_Featured_Clean.csv')
print(f'   Final shape: {df_clean.shape}')
print(f'   Date range : {df_clean["Date"].min().date()} → {df_clean["Date"].max().date()}')
print(f'   Null cells : {df_clean.isnull().sum().sum()}')

---

## 📋 Summary of Cleaning Steps

| Step | Action | Method | Impact |
|------|--------|--------|--------|
| 1 | Fix column names | `str.strip()` | 4 columns renamed |
| 2 | Remove duplicates | `drop_duplicates()` | ~80 rows removed |
| 3 | Fix data types | `pd.to_datetime`, `pd.to_numeric`, category mapping | Date, Volume, Target_Strong fixed |
| 4 | OHLC violations | Swap High/Low | ~25 rows corrected |
| 5 | Negative prices | Set to NaN | ~20 values converted |
| 6 | Outlier capping | IQR Winsorisation (3× IQR) | Extreme spikes capped |
| 7 | RSI out-of-range | Set to NaN | ~40 values nulled |
| 8 | Missing imputation | ffill/bfill, median, mode, zero | All nulls resolved |
| 9 | Sort & reset index | `sort_values('Date')` | Chronological order restored |
| 10 | dtype enforcement | `.astype(int/float)` | Consistent dtypes enforced |